# 00 — From pandas rows to FeatureGraph objects

You know DataFrames and `groupby`. FeatureGraph adds explicit states,
boundaries, object identity, status, provenance, and relations. Protocol labels
below are externally supplied; they are not inferred patient states.


In [ ]:
import numpy as np
import pandas as pd
import featuregraph as fg

observations = pd.DataFrame({
    "sample": np.arange(12),
    "protocol_state": ["baseline"] * 3 + ["task"] * 4 + ["recovery"] * 5,
    "heart_rate": [68, 69, 68, 75, 80, 84, 82, 78, 74, 72, 70, 69],
})
observations


## Compile labels and boundary events

The contract names the existing label column. It does not reinterpret it.


In [ ]:
contract = {
    "version": "state-contract-v1",
    "state_column": "protocol_state",
    "events": {
        "enter_protocol_state": {"type": "enter_label"},
        "exit_protocol_state": {"type": "exit_label"},
    },
}
compiled = fg.compile_states(observations, contract)
compiled.observations


In [ ]:
occurrences = fg.from_state_sequence(
    observations["protocol_state"],
    signal=observations["heart_rate"],
    times=observations["sample"],
    group_id="patient-01",
    dataset="tutorial-healthcare-protocol",
    signal_name="heart_rate",
    signal_unit="beats/minute",
    detector="published_protocol",
    software_version=fg.__version__,
)
objects = occurrences.object_table()
objects


Pandas can summarize after identity exists. FeatureGraph
also retains half-open boundaries, edge status, reconstruction, provenance, and
adjacency. Numeric signals use `fg.transition.Transition` in the next lesson.


In [ ]:
pandas_summary = compiled.observations.groupby(
    "state_occurrence_id", sort=False
).agg(
    state=("state", "first"),
    sample_count=("sample", "size"),
    heart_rate_mean=("heart_rate", "mean"),
).reset_index()
pandas_summary


In [ ]:
assert objects["state_label"].tolist() == ["baseline", "task", "recovery"]
assert objects["sample_count"].tolist() == [3, 4, 5]
assert objects["status"].tolist() == [
    "boundary_truncated", "complete", "boundary_truncated"
]
assert np.array_equal(
    occurrences.reconstruct_states(), observations["protocol_state"].to_numpy()
)
assert len(occurrences.relations) == 2
